In [10]:
import sys
import os
sys.path.insert(0, '/home/jonathan/opf/src')
os.chdir('/home/jonathan/opf')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from tempfile import TemporaryDirectory
import wandb

from opf.test import load_run
from opf.test import data_to_device
from opf.powerflow import BranchParameters

# ── config ────────────────────────────────────────────────────────────────────
RUN_ID = "uomqic9a"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── find best checkpoint via wandb ───────────────────────────────────────────
api = wandb.Api()
run = api.run(f"alelab/opf_param/{RUN_ID}")
artifacts = list(run.logged_artifacts())
model_artifacts = [a for a in artifacts if a.type == "model"]
print("Available model artifacts:")
for a in model_artifacts:
    print(f"  {a.name} aliases={a.aliases} metadata={a.metadata}")

# pick best artifact — use the one with 'best' alias or lowest score
best_artifact = min(model_artifacts, key=lambda a: a.metadata.get("score", float("inf")))
print(f"\nUsing: {best_artifact.name} score={best_artifact.metadata.get('score')}")

with TemporaryDirectory() as tmpdir:
    checkpoint_path = best_artifact.download(root=tmpdir)
    ckpt_file = str(Path(checkpoint_path) / "model.ckpt")
    
    # ── load model ────────────────────────────────────────────────────────────
    dm, opfdual = load_run(
        RUN_ID,
        batch_size=32,
        data_dir="data",
        best=True,
        local_checkpoint_path=ckpt_file,
    )
    opfdual = opfdual.to(DEVICE)
    opfdual.eval()

    print(opfdual.model.training)
    print(opfdual.model_dual.training)
    print(opfdual.model_dual.primal_model.training)

    # ── run inference ─────────────────────────────────────────────────────────
    all_sf_abs = []
    all_st_abs = []
    all_fwd_mult = []
    all_bwd_mult = []
    all_rate_a = []

    with torch.no_grad():
        for data in tqdm(dm.test_dataloader()):
            data = data_to_device(data, DEVICE)
            graph = data.graph
            variables, _, _ = opfdual(data)
            multipliers = opfdual.model_dual.get_multipliers(data)

            num_graphs = graph.num_graphs
            n_branch = graph["branch"].params.shape[0] // num_graphs

            sf_abs = variables.Sf.abs().reshape(num_graphs, n_branch)
            st_abs = variables.St.abs().reshape(num_graphs, n_branch)

            fwd = multipliers["inequality/forward_rate"].reshape(num_graphs, n_branch, 2)
            bwd = multipliers["inequality/backward_rate"].reshape(num_graphs, n_branch, 2)

            branch_params = BranchParameters.from_tensor(graph["branch"]["params"])
            rate_a = branch_params.rate_a.reshape(num_graphs, n_branch)

            all_sf_abs.append(sf_abs.cpu().numpy())
            all_st_abs.append(st_abs.cpu().numpy())
            all_fwd_mult.append(fwd.cpu().numpy())
            all_bwd_mult.append(bwd.cpu().numpy())
            all_rate_a.append(rate_a.cpu().numpy())

# stack
sf_abs   = np.concatenate(all_sf_abs,   axis=0)
st_abs   = np.concatenate(all_st_abs,   axis=0)
fwd_mult = np.concatenate(all_fwd_mult, axis=0)
bwd_mult = np.concatenate(all_bwd_mult, axis=0)
rate_a   = np.concatenate(all_rate_a,   axis=0)

sf_util = sf_abs / (rate_a + 1e-8)
st_util = st_abs / (rate_a + 1e-8)

fwd_upper = fwd_mult[:, :, 1]
bwd_upper = bwd_mult[:, :, 1]

# ── per-branch correlation ────────────────────────────────────────────────────
from scipy.stats import pearsonr

n_branch = sf_abs.shape[1]
fwd_corr = np.zeros(n_branch)
bwd_corr = np.zeros(n_branch)

for b in range(n_branch):
    fwd_corr[b], _ = pearsonr(sf_util[:, b], fwd_upper[:, b])
    bwd_corr[b], _ = pearsonr(st_util[:, b], bwd_upper[:, b])

print("\nForward rate correlations:")
for b, c in enumerate(fwd_corr):
    print(f"  Branch {b:2d}: {c:.4f}")

print("\nBackward rate correlations:")
for b, c in enumerate(bwd_corr):
    print(f"  Branch {b:2d}: {c:.4f}")

best_fwd = np.argmax(fwd_corr)
best_bwd = np.argmax(bwd_corr)
print(f"\nBest forward branch: {best_fwd} (r={fwd_corr[best_fwd]:.4f})")
print(f"Best backward branch: {best_bwd} (r={bwd_corr[best_bwd]:.4f})")

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, branch, util, mult, corr, label in [
    (axes[0], best_fwd, sf_util, fwd_upper, fwd_corr, "Forward"),
    (axes[1], best_bwd, st_util, bwd_upper, bwd_corr, "Backward"),
]:
    x = util[:, branch]
    y = mult[:, branch]
    ax.scatter(x, y, alpha=0.4, s=10, color='steelblue')
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Capacity limit')
    ax.set_xlabel('Line Utilization (|S| / rate_a)')
    ax.set_ylabel('Predicted Dual Multiplier')
    ax.set_title(f'{label} Rate — Branch {branch} (r={corr[branch]:.3f})')
    ax.legend(fancybox=False, edgecolor='black')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('dual_vs_utilization.pdf', bbox_inches='tight', dpi=300)
plt.show()

Available model artifacts:
  model-uomqic9a:v0 aliases=[] metadata={'score': 7.53438138961792, 'original_filename': 'best-4743.ckpt', 'DelayedModelCheckpoint': {'mode': 'min', 'monitor': 'val/invariant', 'save_last': True, 'save_top_k': 5, 'save_weights_only': False, '_every_n_train_steps': 0}}
  model-uomqic9a:v1 aliases=[] metadata={'score': 7.391303539276123, 'original_filename': 'best-4802.ckpt', 'DelayedModelCheckpoint': {'mode': 'min', 'monitor': 'val/invariant', 'save_last': True, 'save_top_k': 5, 'save_weights_only': False, '_every_n_train_steps': 0}}
  model-uomqic9a:v2 aliases=[] metadata={'score': 7.680999279022217, 'original_filename': 'best-4829.ckpt', 'DelayedModelCheckpoint': {'mode': 'min', 'monitor': 'val/invariant', 'save_last': True, 'save_top_k': 5, 'save_weights_only': False, '_every_n_train_steps': 0}}
  model-uomqic9a:v3 aliases=[] metadata={'score': 7.502407073974609, 'original_filename': 'best-4840.ckpt', 'DelayedModelCheckpoint': {'mode': 'min', 'monitor': 'va

wandb:   1 of 1 files downloaded.  


False
False
False


  0%|          | 0/32 [00:00<?, ?it/s]


RuntimeError: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility